# Week 3 fidelity anchor — MedAlpaca-7b on Colab

Reproduces Health-LLM Table 3: **PMData stress, zero-shot MedAlpaca-7b, MAE 0.76 ± 0.1**.

**Before you start**
1. On your Mac, from the repo root: `bash scripts/pack_colab_anchor.sh`
   That writes `results/anchor/colab_anchor.zip` (eval split is gitignored; the zip carries it).
2. Runtime → Change runtime type → **GPU**. Prefer **L4 or A100** (Colab Pro). Free **T4 (16 GB)** is the minimum and may OOM on fp16.
3. Keep this tab focused. ~897 greedy generations; budget **3–6 hours**. Checkpoints go to Drive so a disconnect does not wipe progress.

Do **not** switch to 4-bit unless fp16 OOMs. Quantization is a last-resort deviation and must be disclosed next to the number.

In [ ]:
import torch
assert torch.cuda.is_available(), "Runtime is not a GPU. Runtime → Change runtime type → GPU, then rerun."
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / (1024 ** 3)
print(f"GPU: {props.name}")
print(f"VRAM: {vram_gb:.1f} GB")
if vram_gb < 15:
    raise SystemExit("Need ≥16 GB VRAM for MedAlpaca-7b fp16. This runtime is too small.")
if vram_gb < 20:
    print("T4-class GPU: fp16 may OOM on long prompts. If it dies, stop — do not silently 4-bit.")

In [ ]:
from google.colab import drive, files
from pathlib import Path
import zipfile

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/physio-placebo-anchor")
DRIVE.mkdir(parents=True, exist_ok=True)
print("Drive output:", DRIVE)
print("Upload results/anchor/colab_anchor.zip")
uploaded = files.upload()
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall("/content")
ROOT = Path("/content/physio-placebo")
assert (ROOT / "scripts/run_medalpaca_anchor.py").is_file()
print("extracted", ROOT)

In [ ]:
import os, subprocess, sys
os.chdir("/content/physio-placebo")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers", "accelerate", "sentencepiece"])
print("installed")

In [ ]:
import os, subprocess, sys
os.chdir("/content/physio-placebo")
cmd = [
    sys.executable, "scripts/run_medalpaca_anchor.py",
    "--out-dir", "/content/drive/MyDrive/physio-placebo-anchor",
]
print("running:", " ".join(cmd))
print("Re-run this cell after a disconnect; it resumes from the Drive partials.")
subprocess.check_call(cmd)

When it finishes, `run.json` on Drive has `mae_mean`, `mae_sd`, and `gap_vs_paper`. Download that folder back onto the Mac under `results/anchor/medalpaca/` and we will fill `docs/fidelity-anchor.md` §6.